# VitroVision — 5-Fold Cross-Validation Evaluation
**Metrics:** Cohen's Kappa | MCC | Weighted-F1 | Macro-F1 | AUC (OvR)  
**Bootstrap CI:** n=1000, 95%  
**Model:** EfficientNet-B0 + AdamW layer-wise + Mixup + CosineAnnealingLR

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT     = Path('..').resolve()
DATA_DIR = ROOT / 'shelf_manager' / 'data'

LABELS = ['healthy', 'contaminated', 'dead']
L2I    = {l: i for i, l in enumerate(LABELS)}

jpgs = list(DATA_DIR.rglob('*.jpg')) if DATA_DIR.exists() else []
labeled = [p for p in jpgs if p.stem.split('_')[-1] in L2I]

if len(labeled) == 0:
    print('⚠️  ไม่พบภาพใน shelf_manager/data/')
    print(f'   วางภาพใน : {DATA_DIR}')
    print('   ชื่อไฟล์  : *_healthy.jpg | *_contaminated.jpg | *_dead.jpg')
    raise SystemExit('หยุด: ไม่มีข้อมูล — วางภาพก่อนแล้วค่อย run ใหม่')

print(f'DATA_DIR : {DATA_DIR}')
print(f'พบภาพ    : {len(labeled)} ใบ — พร้อม run K-fold')

## 1 — Load Dataset

In [ ]:
import cv2
import numpy as np
from collections import Counter

def load_dataset(data_dir):
    """label จากชื่อไฟล์: ..._healthy.jpg → 'healthy'"""
    paths, labels = [], []
    for jpg in Path(data_dir).rglob('*.jpg'):
        label = jpg.stem.split('_')[-1]
        if label in L2I:
            paths.append(str(jpg))
            labels.append(label)
    return paths, labels

all_paths, all_labels = load_dataset(DATA_DIR)
counts = Counter(all_labels)
print(f'Dataset: {len(all_paths)} ภาพ')
for cls in LABELS:
    print(f'  {cls:15s}: {counts.get(cls, 0):4d} ภาพ')

if min(counts.get(c, 0) for c in LABELS) < 5:
    print('\n⚠️  บาง class มีภาพน้อยกว่า 5 — K-fold อาจ stratify ไม่ได้')

## 2 — Setup: KFold, Metrics, Transforms

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    cohen_kappa_score, matthews_corrcoef,
    f1_score, roc_auc_score,
    confusion_matrix, classification_report,
)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(ROOT))
from vitro_vision.transforms import get_train_transforms, get_val_transforms

IMG_SIZE = 224
BATCH    = 8
LR       = 1e-4
EP_HEAD  = 15
EP_FULL  = 25
N_SPLITS = 5
N_BOOT   = 1000
DEVICE   = 'cpu'

CKPT_DIR = ROOT / 'models' / 'checkpoints' / 'kfold'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
print(f'StratifiedKFold : {N_SPLITS} folds')
print(f'Epochs          : head={EP_HEAD}, full={EP_FULL}')
print(f'Metrics per fold: Kappa, MCC, Weighted-F1, Macro-F1, AUC(OvR)')

## 3 — K-Fold Training Loop

In [ ]:
class BottleDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths     = paths
        self.labels    = [L2I[l] for l in labels]
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = cv2.cvtColor(cv2.imread(self.paths[idx]), cv2.COLOR_BGR2RGB)
        return self.transform(image=img)['image'], self.labels[idx]


def _train_fold(tr_paths, tr_labels, vl_paths, vl_labels, fold_id):
    """Train EfficientNet-B0 บน 1 fold — คืน (trues, preds, probs)"""
    tr_loader = DataLoader(
        BottleDataset(tr_paths, tr_labels, get_train_transforms(IMG_SIZE)),
        batch_size=BATCH, shuffle=True, num_workers=0)
    vl_loader = DataLoader(
        BottleDataset(vl_paths, vl_labels, get_val_transforms(IMG_SIZE)),
        batch_size=BATCH, shuffle=False, num_workers=0)

    model     = timm.create_model('efficientnet_b0', pretrained=True,
                                   num_classes=3, drop_rate=0.4, drop_path_rate=0.15)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    def run_epoch(loader, opt=None):
        model.train() if opt else model.eval()
        ctx = torch.enable_grad() if opt else torch.no_grad()
        with ctx:
            for imgs, tgts in loader:
                out  = model(imgs)
                loss = criterion(out, tgts)
                if opt: opt.zero_grad(); loss.backward(); opt.step()

    # Phase 1 — head only
    for p in model.parameters(): p.requires_grad = False
    for p in model.classifier.parameters(): p.requires_grad = True
    opt1 = torch.optim.AdamW(model.classifier.parameters(), lr=LR, weight_decay=1e-2)
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=EP_HEAD, eta_min=LR*0.01)
    for ep in range(1, EP_HEAD + 1):
        run_epoch(tr_loader, opt1); sch1.step()
        if ep % 5 == 0: print(f'    Phase1 ep {ep}/{EP_HEAD}')

    # Phase 2 — fine-tune all (AdamW layer-wise)
    for p in model.parameters(): p.requires_grad = True
    opt2 = torch.optim.AdamW([
        {'params': [p for n, p in model.named_parameters() if 'classifier' not in n], 'lr': 1e-5},
        {'params': model.classifier.parameters(), 'lr': 1e-4},
    ], weight_decay=1e-2)
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=EP_FULL, eta_min=1e-7)
    for ep in range(1, EP_FULL + 1):
        run_epoch(tr_loader, opt2); sch2.step()
        if ep % 5 == 0: print(f'    Phase2 ep {ep}/{EP_FULL}')

    torch.save(model, CKPT_DIR / f'fold{fold_id}.pt')

    # Evaluate on val
    model.eval()
    preds, probs_list, trues = [], [], []
    with torch.no_grad():
        for imgs, tgts in vl_loader:
            p = F.softmax(model(imgs), dim=1)
            preds.extend(p.argmax(1).tolist())
            probs_list.extend(p.tolist())
            trues.extend(tgts.tolist())
    return trues, preds, probs_list


# ── Run K-Fold ───────────────────────────────────────────────────────
paths_arr  = np.array(all_paths)
labels_arr = np.array(all_labels)
label_idx  = np.array([L2I[l] for l in all_labels])

fold_results = []
for fold, (tr_idx, vl_idx) in enumerate(skf.split(paths_arr, label_idx), 1):
    print(f'\n=== Fold {fold}/{N_SPLITS} ===')
    tr_p = paths_arr[tr_idx].tolist();  tr_l = labels_arr[tr_idx].tolist()
    vl_p = paths_arr[vl_idx].tolist();  vl_l = labels_arr[vl_idx].tolist()
    print(f'  Train {len(tr_p)} | Val {len(vl_p)} | dist: {dict(Counter(vl_l))}')

    trues, preds, probs = _train_fold(tr_p, tr_l, vl_p, vl_l, fold)

    kappa = cohen_kappa_score(trues, preds)
    mcc   = matthews_corrcoef(trues, preds)
    wf1   = f1_score(trues, preds, average='weighted', zero_division=0)
    maf1  = f1_score(trues, preds, average='macro',    zero_division=0)
    try:
        auc = roc_auc_score(trues, probs, multi_class='ovr', average='macro')
    except Exception:
        auc = float('nan')

    fold_results.append({
        'fold': fold, 'kappa': kappa, 'mcc': mcc,
        'weighted_f1': wf1, 'macro_f1': maf1, 'auc_ovr': auc,
        'trues': trues, 'preds': preds, 'probs': probs,
    })
    print(f'  Kappa={kappa:.3f} | MCC={mcc:.3f} | wF1={wf1:.3f} | mF1={maf1:.3f} | AUC={auc:.3f}')

## 4 — Aggregate Results + Bootstrap CI

In [ ]:
def bootstrap_ci(trues, preds, metric_fn, n=1000, ci=95):
    """Bootstrap CI สำหรับ metric ใดก็ได้ที่รับ (y_true, y_pred)"""
    arr = np.array(list(zip(trues, preds)))
    scores = []
    for _ in range(n):
        idx = np.random.choice(len(arr), len(arr), replace=True)
        s   = arr[idx]
        try:
            scores.append(metric_fn(s[:, 0], s[:, 1]))
        except Exception:
            pass
    lo = np.percentile(scores, (100 - ci) / 2)
    hi = np.percentile(scores, 100 - (100 - ci) / 2)
    return float(np.mean(scores)), float(lo), float(hi)


# รวม predictions ทุก fold (pooled)
all_trues = [t for f in fold_results for t in f['trues']]
all_preds = [p for f in fold_results for p in f['preds']]

METRICS = ['kappa', 'mcc', 'weighted_f1', 'macro_f1', 'auc_ovr']
summary = {}
for m in METRICS:
    vals = [f[m] for f in fold_results]
    summary[m] = {'mean': float(np.mean(vals)), 'std': float(np.std(vals))}

np.random.seed(42)
kappa_ci = bootstrap_ci(all_trues, all_preds, cohen_kappa_score, n=N_BOOT)
f1_ci    = bootstrap_ci(all_trues, all_preds,
                        lambda t, p: f1_score(t, p, average='weighted', zero_division=0),
                        n=N_BOOT)
mcc_ci   = bootstrap_ci(all_trues, all_preds, matthews_corrcoef, n=N_BOOT)

print('=== K-Fold Summary (mean \u00b1 std across 5 folds) ===')
for m in METRICS:
    print(f'  {m:15s}: {summary[m]["mean"]:.3f} \u00b1 {summary[m]["std"]:.3f}')

print(f'\nBootstrap CI (pooled, n={N_BOOT}, 95%):')
print(f'  Kappa  : {kappa_ci[0]:.3f}  [{kappa_ci[1]:.3f}\u2013{kappa_ci[2]:.3f}]')
print(f'  wF1    : {f1_ci[0]:.3f}  [{f1_ci[1]:.3f}\u2013{f1_ci[2]:.3f}]')
print(f'  MCC    : {mcc_ci[0]:.3f}  [{mcc_ci[1]:.3f}\u2013{mcc_ci[2]:.3f}]')

# Confusion matrix รวม
cm = confusion_matrix(all_trues, all_preds, labels=list(range(3)))
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix (pooled 5-fold)')
plt.tight_layout(); plt.show()

## 5 — McNemar's Test vs Random Baseline

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

def mcnemar_vs_baseline(y_true, y_pred, random_seed=42):
    """McNemar's test เปรียบ model vs stratified random baseline"""
    rng = np.random.default_rng(random_seed)
    classes, counts = np.unique(y_true, return_counts=True)
    probs      = counts / counts.sum()
    y_baseline = rng.choice(classes, size=len(y_true), p=probs)
    m     = (np.array(y_pred)     == np.array(y_true)).astype(int)
    b_arr = (np.array(y_baseline) == np.array(y_true)).astype(int)
    b = int(np.sum((m == 1) & (b_arr == 0)))
    c = int(np.sum((m == 0) & (b_arr == 1)))
    table  = [[int(np.sum((m == 1) & (b_arr == 1))), b],
               [c, int(np.sum((m == 0) & (b_arr == 0)))]]
    result = mcnemar(table, exact=(b + c) < 25, correction=True)
    return {
        'statistic':   result.statistic,
        'pvalue':      result.pvalue,
        'b':           b,
        'c':           c,
        'significant': result.pvalue < 0.05,
    }


mc_result = mcnemar_vs_baseline(all_trues, all_preds)
print("McNemar's Test — Model vs Stratified Random Baseline")
print(f'  b (model correct, baseline wrong) : {mc_result["b"]}')
print(f'  c (model wrong,   baseline correct): {mc_result["c"]}')
print(f'  statistic : {mc_result["statistic"]:.4f}')
print(f'  p-value   : {mc_result["pvalue"]:.4f}')
print(f'  significant (p<0.05): {mc_result["significant"]}')

## 6 — Report Table

In [ ]:
# Per-fold table
rows = []
for f in fold_results:
    rows.append({
        'Fold':        f['fold'],
        'Kappa':       round(f['kappa'],       3),
        'MCC':         round(f['mcc'],         3),
        'Weighted-F1': round(f['weighted_f1'], 3),
        'Macro-F1':    round(f['macro_f1'],    3),
        'AUC (OvR)':   round(f['auc_ovr'],     3),
    })

# Summary row
rows.append({
    'Fold':        'Mean\u00b1Std',
    'Kappa':       f"{summary['kappa']['mean']:.3f}\u00b1{summary['kappa']['std']:.3f}",
    'MCC':         f"{summary['mcc']['mean']:.3f}\u00b1{summary['mcc']['std']:.3f}",
    'Weighted-F1': f"{summary['weighted_f1']['mean']:.3f}\u00b1{summary['weighted_f1']['std']:.3f}",
    'Macro-F1':    f"{summary['macro_f1']['mean']:.3f}\u00b1{summary['macro_f1']['std']:.3f}",
    'AUC (OvR)':   f"{summary['auc_ovr']['mean']:.3f}\u00b1{summary['auc_ovr']['std']:.3f}",
})
rows.append({
    'Fold':        'Boot CI',
    'Kappa':       f"[{kappa_ci[1]:.3f}\u2013{kappa_ci[2]:.3f}]",
    'MCC':         f"[{mcc_ci[1]:.3f}\u2013{mcc_ci[2]:.3f}]",
    'Weighted-F1': f"[{f1_ci[1]:.3f}\u2013{f1_ci[2]:.3f}]",
    'Macro-F1':    '',
    'AUC (OvR)':   '',
})

df = pd.DataFrame(rows).set_index('Fold')
display(df.style.set_caption('VitroVision 5-Fold Cross-Validation Results'))